# Module 2: Data Processing — Ejercicios Prácticos

**Objetivos:**
- Detectar y manejar valores faltantes
- Identificar y tratar outliers
- Aplicar scaling y normalización
- Encoding de variables categóricas

**Tiempo:** 30 min
**Dataset:** Dataset real (UCI o simulado)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

## Ejercicio 1: Crear Dataset Sucio

In [ ]:
np.random.seed(42)

# Dataset con problemas reales
data = {
    'Age': [25, 32, np.nan, 45, 28, np.nan, 35, 42, 50, 38],
    'Income': [35000, 52000, 48000, 95000, 41000, 58000, 65000, 1500000, 72000, 55000],  # 95000 es outlier
    'Education': ['High School', 'Bachelor', 'Bachelor', np.nan, 'Master', 'High School', 'Bachelor', 'Master', 'PhD', 'Bachelor'],
    'Years_Experience': [2, 8, np.nan, 20, 5, 12, 10, 18, 25, 14],
    'Employed': ['Yes', 'Yes', 'No', 'Yes', np.nan, 'Yes', 'No', 'Yes', 'Yes', 'Yes']
}

df = pd.DataFrame(data)
print("Dataset Original (con problemas):")
print(df)
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDatos: {df.shape[0]} filas, {df.shape[1]} columnas")

## Ejercicio 2: Detectar y Manejar Missing Values

In [ ]:
# Visualizar missing
missing_pct = (df.isnull().sum() / len(df)) * 100
print("Porcentaje de valores faltantes por columna:")
print(missing_pct[missing_pct > 0])

# Opción 1: Eliminar filas
df_dropped = df.dropna()
print(f"\nOpción 1 (Eliminar): {df_dropped.shape[0]} filas restantes")

# Opción 2: Imputación con media
df_imputed = df.copy()
imputer = SimpleImputer(strategy='mean')
df_imputed[['Age', 'Years_Experience']] = imputer.fit_transform(df[['Age', 'Years_Experience']])
# Para categóricas
df_imputed['Education'].fillna(df_imputed['Education'].mode()[0], inplace=True)
df_imputed['Employed'].fillna(df_imputed['Employed'].mode()[0], inplace=True)

print(f"\nOpción 2 (Imputar): {df_imputed.shape[0]} filas")
print("\nDataset después de imputación:")
print(df_imputed)
print(f"\nMissing values después: {df_imputed.isnull().sum().sum()}")

## Ejercicio 3: Detectar Outliers

In [ ]:
# Método 1: IQR
Q1 = df_imputed['Income'].quantile(0.25)
Q3 = df_imputed['Income'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df_imputed[(df_imputed['Income'] < lower_bound) | (df_imputed['Income'] > upper_bound)]
print(f"Outliers detectados (IQR):")
print(outliers_iqr)

# Método 2: Z-score
z_scores = np.abs(stats.zscore(df_imputed['Income']))
outliers_z = df_imputed[z_scores > 3]
print(f"\nOutliers detectados (Z-score > 3):")
print(outliers_z)

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter plot
axes[0].scatter(range(len(df_imputed)), df_imputed['Income'])
axes[0].axhline(y=upper_bound, color='r', linestyle='--', label='IQR bounds')
axes[0].axhline(y=lower_bound, color='r', linestyle='--')
axes[0].set_title('Income (IQR method)')
axes[0].set_ylabel('Income')
axes[0].legend()

# Boxplot
axes[1].boxplot(df_imputed['Income'])
axes[1].set_title('Boxplot de Income')
axes[1].set_ylabel('Income')

# Histograma
axes[2].hist(df_imputed['Income'], bins=20, edgecolor='black')
axes[2].set_title('Distribución de Income')
axes[2].set_xlabel('Income')

plt.tight_layout()
plt.show()

## Ejercicio 4: Tratar Outliers

In [ ]:
df_cleaned = df_imputed.copy()

# Opción 1: Eliminar
df_no_outliers = df_cleaned[(df_cleaned['Income'] >= lower_bound) & (df_cleaned['Income'] <= upper_bound)]
print(f"Opción 1 (Eliminar): {df_no_outliers.shape[0]} filas")

# Opción 2: Capping (Winsorization)
df_capped = df_cleaned.copy()
df_capped['Income_capped'] = df_capped['Income'].clip(lower=lower_bound, upper=upper_bound)
print(f"\nOpción 2 (Capping): {df_capped.shape[0]} filas")
print(f"Income original (max): {df_capped['Income'].max()}")
print(f"Income capped (max): {df_capped['Income_capped'].max()}")

# Opción 3: Transformación log
df_log = df_cleaned.copy()
df_log['Income_log'] = np.log1p(df_log['Income'])

# Visualizar
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df_cleaned['Income'], bins=15, edgecolor='black')
axes[0, 0].set_title('Original')
axes[0, 0].set_xlabel('Income')

axes[0, 1].hist(df_capped['Income_capped'], bins=15, edgecolor='black')
axes[0, 1].set_title('Capped')
axes[0, 1].set_xlabel('Income')

axes[1, 0].hist(df_log['Income_log'], bins=15, edgecolor='black')
axes[1, 0].set_title('Log Transform')
axes[1, 0].set_xlabel('Log(Income)')

axes[1, 1].hist(df_no_outliers['Income'], bins=15, edgecolor='black')
axes[1, 1].set_title('Outliers Removed')
axes[1, 1].set_xlabel('Income')

plt.tight_layout()
plt.show()

## Ejercicio 5: Scaling y Normalización

In [ ]:
df_processed = df_capped.copy()

# StandardScaler
scaler_std = StandardScaler()
df_processed['Age_std'] = scaler_std.fit_transform(df_processed[['Age']])
df_processed['Income_std'] = scaler_std.fit_transform(df_processed[['Income']])

# MinMaxScaler
scaler_minmax = MinMaxScaler()
df_processed['Age_minmax'] = scaler_minmax.fit_transform(df_processed[['Age']])
df_processed['Income_minmax'] = scaler_minmax.fit_transform(df_processed[['Income']])

print("Comparación de Scaling:")
print(df_processed[['Age', 'Age_std', 'Age_minmax']].head())

# Visualizar
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Age original
axes[0, 0].hist(df_processed['Age'], bins=10, edgecolor='black')
axes[0, 0].set_title('Age (Original)')
axes[0, 0].set_ylabel('Frequency')

# Age StandardScaler
axes[0, 1].hist(df_processed['Age_std'], bins=10, edgecolor='black')
axes[0, 1].set_title('Age (StandardScaler)')
axes[0, 1].set_ylabel('Frequency')

# Age MinMaxScaler
axes[0, 2].hist(df_processed['Age_minmax'], bins=10, edgecolor='black')
axes[0, 2].set_title('Age (MinMaxScaler)')
axes[0, 2].set_ylabel('Frequency')

# Income original
axes[1, 0].hist(df_processed['Income'], bins=10, edgecolor='black')
axes[1, 0].set_title('Income (Original)')
axes[1, 0].set_ylabel('Frequency')

# Income StandardScaler
axes[1, 1].hist(df_processed['Income_std'], bins=10, edgecolor='black')
axes[1, 1].set_title('Income (StandardScaler)')
axes[1, 1].set_ylabel('Frequency')

# Income MinMaxScaler
axes[1, 2].hist(df_processed['Income_minmax'], bins=10, edgecolor='black')
axes[1, 2].set_title('Income (MinMaxScaler)')
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"\nRango StandardScaler: [{df_processed['Age_std'].min():.2f}, {df_processed['Age_std'].max():.2f}]")
print(f"Rango MinMaxScaler: [{df_processed['Age_minmax'].min():.2f}, {df_processed['Age_minmax'].max():.2f}]")

## Ejercicio 6: Encoding de Categóricas

In [ ]:
# One-Hot Encoding
df_onehot = df_processed.copy()
education_onehot = pd.get_dummies(df_onehot['Education'], prefix='Education', drop_first=False)
df_onehot = pd.concat([df_onehot, education_onehot], axis=1)

print("One-Hot Encoding de Education:")
print(df_onehot[['Education', 'Education_Bachelor', 'Education_High School', 'Education_Master', 'Education_PhD']])

# Label Encoding
df_label = df_processed.copy()
le = LabelEncoder()
df_label['Education_encoded'] = le.fit_transform(df_label['Education'])
df_label['Employed_encoded'] = LabelEncoder().fit_transform(df_label['Employed'])

print("\nLabel Encoding:")
print(df_label[['Education', 'Education_encoded']])
print(f"\nMapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Target Encoding (simulado)
df_target = df_processed.copy()
# Crear variable target simulada
df_target['salary_high'] = (df_target['Income'] > df_target['Income'].median()).astype(int)
target_encoded = df_target.groupby('Education')['salary_high'].mean()
df_target['Education_target'] = df_target['Education'].map(target_encoded)

print("\nTarget Encoding (Media de salary_high por Education):")
print(df_target[['Education', 'salary_high', 'Education_target']])
print(f"\nMapping: {target_encoded.to_dict()}")

## Resumen

In [ ]:
print("="*60)
print("RESUMEN: DATA PROCESSING")
print("="*60)
print()
print("1. MISSING VALUES:")
print("   - <5%: Eliminar")
print("   - 5-20%: Imputación simple (media/mediana/moda)")
print("   - >20%: Imputación avanzada (KNN, MICE)")
print()
print("2. OUTLIERS:")
print("   - IQR, Z-score, Isolation Forest para detectar")
print("   - Eliminar, capping, o transformación")
print()
print("3. SCALING:")
print("   - StandardScaler: distribución normal")
print("   - MinMaxScaler: rango 0-1")
print("   - RobustScaler: muchos outliers")
print()
print("4. ENCODING:")
print("   - One-Hot: pocas categorías, árboles")
print("   - Label: muchas categorías, regresión lineal")
print("   - Target: relación fuerte con variable objetivo")
print()
print("="*60)